## Local Agent with LMStudio and Mistral Small

In this example we will learn how to use fully local LLMs, implementing the latest `mistral-small-3.1`. An incredibly competent LLM fully competent of powering agentic workflows with tool calling.

In [1]:
!pip install -qU \
    litellm==1.63.14 \
    serpapi==0.1.5 \
    google-search-results==2.4.2

zsh:1: command not found: pip


We will be using LM Studio to host Mistral Small 3.1 locally, installation instructions can be found [here](https://lmstudio.ai/download).

LM Studio (on Mac) supports all GGUF quantized models. That means that we must use [`lmstudio-community/Mistral-Small-3.1-24B-Instruct-2503-GGUF`](https://huggingface.co/lmstudio-community/Mistral-Small-3.1-24B-Instruct-2503-GGUF) which can be downloaded for LM Studio [here](https://model.lmstudio.ai/download/lmstudio-community/Mistral-Small-3.1-24B-Instruct-2503-GGUF).

## Using Mistral Small

Once the model has been downloaded we can select **Start server on port 1234** in our LM Studio interface (accessible by clicking the icon in the taskbar) and load our chosen model. Then we confirm LM Studio is accessible like so:

In [1]:
!curl http://localhost:1234/v1/models

{
  "data": [
    {
      "id": "mistral-small-3.1-24b-instruct-2503",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "text-embedding-nomic-embed-text-v1.5",
      "object": "model",
      "owned_by": "organization_owner"
    }
  ],
  "object": "list"
}

We can take the model name from above and insert it into our `model` parameter below (after `lm_studio/`):

In [2]:
from litellm import completion
import os

# set to the port LM studio is using, default is 1234
os.environ["LM_STUDIO_API_BASE"] = "http://localhost:1234/v1"

response = completion(
    model="lm_studio/mistral-small-3.1-24b-instruct-2503",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key"  # need a dummy API key
)
print(response)

ModelResponse(id='chatcmpl-8da4kz2rxr6kz61kjtefj', created=1743228890, model='lm_studio/mistral-small-3.1-24b-instruct-2503', object='chat.completion', system_fingerprint='mistral-small-3.1-24b-instruct-2503', choices=[Choices(finish_reason='stop', index=0, message=Message(content="Hello! I'm functioning as intended, thank you. How can I assist you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None, 'annotations': None}))], usage=Usage(completion_tokens=18, prompt_tokens=167, total_tokens=185, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})


## Async Streaming

Let's see how we can implement asynchronous streaming via LiteLLM.

In [3]:
from litellm import acompletion

response = await acompletion(
    model="lm_studio/mistral-small-3.1-24b-instruct-2503",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    stream=True
)

async for chunk in response:
    print(chunk)

ModelResponseStream(id='chatcmpl-975c4fe7-d491-452c-85ff-c978995393c0', created=1743228895, model='mistral-small-3.1-24b-instruct-2503', object='chat.completion.chunk', system_fingerprint='mistral-small-3.1-24b-instruct-2503', choices=[StreamingChoices(finish_reason=None, index=0, delta=Delta(provider_specific_fields=None, refusal=None, content='Hello', role='assistant', function_call=None, tool_calls=None, audio=None), logprobs=None)], provider_specific_fields=None, stream_options=None, citations=None)
ModelResponseStream(id='chatcmpl-975c4fe7-d491-452c-85ff-c978995393c0', created=1743228895, model='mistral-small-3.1-24b-instruct-2503', object='chat.completion.chunk', system_fingerprint='mistral-small-3.1-24b-instruct-2503', choices=[StreamingChoices(finish_reason=None, index=0, delta=Delta(provider_specific_fields=None, refusal=None, content='!', role=None, function_call=None, tool_calls=None, audio=None), logprobs=None)], provider_specific_fields=None, stream_options=None, citations

We can parse out the output here like so:

In [4]:
response = await acompletion(
    model="lm_studio/mistral-small-3.1-24b-instruct-2503",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    stream=True
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

Hello! I'm here and ready to assist you. How can I help you today? If you're looking for information or need help with something specific, just let me know!

## Tool Calls

Tool calling is naturally a big part of what makes agents useful — let's see how we can do this with Mistral small. First, we would usually check if our model can use function calling via `supports_function_calling`:

In [5]:
from litellm import supports_function_calling

supports_function_calling("lm_studio/mistral-small-3.1-24b-instruct-2503")

False

This tells us we _cannot_ use function calling, but this is not entirely true. We _can_ use function calling but we must use LM Studio's OpenAI chat completion endpoint. This endpoint _does not_ use OpenAI, but simply replicates the pattern of OpenAI's chat completion endpoint. The method for calling this is slightly different, synchronously we do it like so:

In [6]:
response = completion(
    model="openai/mistral-small-3.1-24b-instruct-2503",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

response

ModelResponse(id='chatcmpl-iaxd34f71dg3kbnaz6xz', created=1743228901, model='mistral-small-3.1-24b-instruct-2503', object='chat.completion', system_fingerprint='mistral-small-3.1-24b-instruct-2503', choices=[Choices(finish_reason='stop', index=0, message=Message(content="Hello! I'm here and ready to assist you. How can I help you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None, 'annotations': None}))], usage=Usage(completion_tokens=18, prompt_tokens=167, total_tokens=185, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})

All we changed here is:

* We added a dummy `api_key`
* We override the default `openai/` URL via `base_url`
* We swap `lm_studio/` for `openai/` in the `model` name.

The pattern is the same for async streaming:

In [7]:
response = await acompletion(
    model="openai/mistral-small-3.1-24b-instruct-2503",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    stream=True,
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

Hello! I'm just a computer program, so I don't have feelings, but thank you for asking. How can I assist you today? If you're up for it, let's chat about something interesting or answer any questions you might have.

Now we have that, let's begin by defining a few tools. The first of those will enable access to the web for our agent via the SerpAPI — for which you can get a free API key [here](https://serpapi.com/dashboard):

In [11]:
from getpass import getpass
import aiohttp

SERPAPI_API_KEY = getpass("Enter your SerpAPI API key: ")

# define our search parameters
params = {
    "api_key": SERPAPI_API_KEY,
    "engine": "google",
    "q": "latest world news"
}

async with aiohttp.ClientSession() as session:
    async with session.get(
        "https://serpapi.com/search",
        params=params
    ) as response:
        results = await response.json()

results["organic_results"]

[{'position': 1,
  'title': 'World | Latest News & Updates',
  'link': 'https://www.bbc.com/news/world',
  'redirect_link': 'https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.bbc.com/news/world&ved=2ahUKEwifhfmS1K6MAxVFATQIHUZgEa0QFnoECC0QAQ',
  'displayed_link': 'https://www.bbc.com › news › world',
  'favicon': 'https://serpapi.com/searches/67e7927182405616dcf35035/images/16c8f2982b167f589c032bebd52ffcaac23c298b281c232418cab8d216c47a55.png',
  'date': 'New57 minutes ago',
  'snippet': 'Four killed in mass Russian drone attack on Dnipro - Ukraine. Another 19 people are injured, as a restaurant and several buildings are set ablaze in the city, ...',
  'snippet_highlighted_words': ['Four killed in mass Russian drone attack on Dnipro - Ukraine'],
  'sitelinks': {'inline': [{'title': 'BBC World',
     'link': 'https://www.bbc.com/news/world_radio_and_tv'},
    {'title': 'Europe', 'link': 'https://www.bbc.com/news/world/europe'},
    {'title': 'Middle East',
   

This is our _async_ call to perform Google searches via SerpAPI, the results are pretty heavy so we can organize them with pydantic like so:

In [33]:
from pydantic import BaseModel

class Article(BaseModel):
    title: str
    source: str
    link: str
    snippet: str

    @classmethod
    def from_serpapi_result(cls, result: dict) -> "Article":
        return cls(
            title=result["title"],
            source=result["source"],
            link=result["link"],
            snippet=result["snippet"],
        )
    
    def __str__(self) -> str:
        return f"## {self.title} - ({self.source})\n_{self.link}_\n{self.snippet}\n"
    
articles = [Article.from_serpapi_result(result) for result in results["organic_results"]]
articles

[Article(title='World | Latest News & Updates', source='BBC', link='https://www.bbc.com/news/world', snippet='Four killed in mass Russian drone attack on Dnipro - Ukraine. Another 19 people are injured, as a restaurant and several buildings are set ablaze in the city, ...'),
 Article(title='World news - breaking news, video, headlines and opinion', source='CNN', link='https://www.cnn.com/world', snippet='March 28, 2025: Magnitude 7.7 earthquake in Myanmar · Afghan pilots who fought in 20-year war against Taliban in limbo after Trump blocks US resettlement plans.'),
 Article(title='World News | Latest Top Stories', source='Reuters', link='https://www.reuters.com/world/', snippet="World · Myanmar quake death toll nears 700 as international aid starts to arrive · In Taiwan's Little Myanmar, fear for quake affected relatives · Vance accuses ..."),
 Article(title='CNN: Breaking News, Latest News and Videos', source='CNN', link='https://www.cnn.com/', snippet='View the latest news and breaki

We format all of this into a single function that our LLM will be able to call:

In [34]:
async def web_search(query: str) -> list[Article]:
    """Use this function to search the web for information. Provide natural language to the
    query with as much context as possible to get the best results.
    """
    params = {
        "api_key": SERPAPI_API_KEY,
        "engine": "google",
        "q": query
    }
    
    async with aiohttp.ClientSession() as session:
        async with session.get(
            "https://serpapi.com/search",
            params=params
        ) as response:
            results = await response.json()
            
    articles = [Article.from_serpapi_result(result) for result in results["organic_results"]]
    articles = "\n".join([str(article) for article in articles])
    return articles

Then we parse these tools into a list of function schemas that our LLM will be able to read:

In [35]:
from graphai.utils import get_schemas

tools = get_schemas(callables=[web_search], format="default")
tools

[{'type': 'function',
  'function': {'name': 'web_search',
   'description': 'Use this function to search the web for information. Provide natural language to the\nquery with as much context as possible to get the best results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'No description available.'}},
    'required': ['query']}}}]

In [48]:
query = {"role": "user", "content": "tell me about the latest world news"}

response = completion(
    model="openai/mistral-small-3.1-24b-instruct-2503",
    messages=[query],
    tools=tools,
    tool_choice="auto",
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

(
    response.choices[0].message.tool_calls[0].function.name,
    response.choices[0].message.tool_calls[0].function.arguments,
)

10:47:09 - LiteLLM:INFO: utils.py:3035 - 
LiteLLM completion() model= mistral-small-3.1-24b-instruct-2503; provider = openai
2025-03-29 10:47:09 - LiteLLM - INFO - utils.py:3035 - _check_valid_arg() - 
LiteLLM completion() model= mistral-small-3.1-24b-instruct-2503; provider = openai
2025-03-29 10:47:10 - httpx - INFO - _client.py:1025 - _send_single_request() - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
10:47:10 - LiteLLM:INFO: utils.py:1165 - Wrapper: Completed Call, calling success_handler
2025-03-29 10:47:10 - LiteLLM - INFO - utils.py:1165 - wrapper() - Wrapper: Completed Call, calling success_handler
10:47:10 - LiteLLM:INFO: cost_calculator.py:588 - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503
2025-03-29 10:47:10 - LiteLLM - INFO - cost_calculator.py:588 - completion_cost() - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503


('web_search', '{"query":"latest world news"}')

Our LLM has generated the tool choice and input parameters for our tool but we have not executed the tool, we must handle that ourselves. To do so we will create a mapping from tool names to their functions.

In [49]:
tool_map = {
    "web_search": web_search
    # when using multiple tools, we would add them here
}

Now we execute the tool like so:

In [50]:
from IPython.display import Markdown, display

tool_out = await tool_map[response.choices[0].message.tool_calls[0].function.name](
    response.choices[0].message.tool_calls[0].function.arguments
)
display(Markdown(tool_out))

## World News - Latest and Breaking Coverage - (Yahoo News)
_https://news.yahoo.com/world/_
The latest world news and headlines from Yahoo News and international ... Search query. Advertisement. World. Business·Yahoo Finance. How the rollout of ...

## Top & Breaking World News Today - (AP News)
_https://apnews.com/world-news_
Español · Standards · Quizzes · Press Releases · My Account. Search Query Submit Search. Show Search. Submit Search. World · Mideast Wars · Russia-Ukraine War ...

## World News - Latest and Breaking Coverage - (Yahoo)
_https://www.yahoo.com/news/world/_
The latest world news and headlines from Yahoo News and international ... Search query. Advertisement. World. World·Associated Press. A mob in southern ...

## Latest World News Today | International News Headlines - (Mint)
_https://www.livemint.com/news/world_
Query, Suggestion. Your Message. footLogo. Connect with us: footLogo. trending stories. Bihar Board Result 2025 Myanmar Earthquake ...

## World News - (Wion)
_https://www.wionews.com/world?page=180_
Get top and latest World News - Read Breaking World News and World News Headlines ... Responding to a query from WION, US Ambassador to India, Eric Garcetti ...

## Latest World News, International News | Breaking World News - (The Express Tribune)
_https://tribune.com.pk/WORLD/archives?page=1049_
Find the latest world news and International news headlines today ... query. it took indian army 15 months to prepare for cross loc surgical strike ...

## Latest World News Updates & Newspaper Headlines - (Pulse Nigeria)
_https://www.pulse.ng/news/world/query.%7B%22page%22%3A%2238%22%7D_
World · US halts $175m funding to University of Pennsylvania over transgender sports policy · US, Israel explore relocating Palestinians in Gaza to Somalia, Sudan ...

## US drops $10 million bounty on Taliban leader Sirajuddin ... - (Mint)
_https://www.livemint.com/news/world/latest-world-news-on-march-23-2025-live-updates-11742669980996.html_
Query, Suggestion. Your Message. footLogo. Connect with us: footLogo. trending stories. Indian stock market NCC share ...

## NDTV.com: Get Latest News, India News, Breaking News ... - (NDTV)
_https://www.ndtv.com/_
CSK Coach Snaps At Journalist Over 'Outdated Way Of Playing Cricket' Query · CSK Coach Snaps At Journalist Over "Outdated Way Of Playing Cricket" Query.

## World News Live Today February 10, 2025: Donald Trump ... - (Hindustan Times)
_https://www.hindustantimes.com/world-news/world-news-live-latest-updates-on-politics-economy-conflicts-climate-change-today-february-10-2025-101739131295462.html_
US News Live : Krelim's cryptic answer to 'Trump-Putin talks to end Ukraine war' query. Russia had denied previous reports of 'private ...


We then format this and the initial tool call from our LLM into messages, and feed them back into our LLM for a final response.

In [51]:
tool_call = {"role": "assistant", "content": response.choices[0].message.content, "tool_calls": response.choices[0].message.tool_calls, "tool_call_id": response.choices[0].message.tool_calls[0].id}
tool_exec = {"role": "tool", "content": tool_out, "tool_call_id": response.choices[0].message.tool_calls[0].id}

In [52]:
messages = [query, tool_call, tool_exec]

response = completion(
    model="openai/mistral-small-3.1-24b-instruct-2503",
    messages=messages,
    tool_choice="auto",
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
    tools=tools
)
response

10:48:03 - LiteLLM:INFO: utils.py:3035 - 
LiteLLM completion() model= mistral-small-3.1-24b-instruct-2503; provider = openai
2025-03-29 10:48:03 - LiteLLM - INFO - utils.py:3035 - _check_valid_arg() - 
LiteLLM completion() model= mistral-small-3.1-24b-instruct-2503; provider = openai
2025-03-29 10:48:09 - httpx - INFO - _client.py:1025 - _send_single_request() - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
10:48:09 - LiteLLM:INFO: utils.py:1165 - Wrapper: Completed Call, calling success_handler
2025-03-29 10:48:09 - LiteLLM - INFO - utils.py:1165 - wrapper() - Wrapper: Completed Call, calling success_handler
10:48:09 - LiteLLM:INFO: cost_calculator.py:588 - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503
2025-03-29 10:48:09 - LiteLLM - INFO - cost_calculator.py:588 - completion_cost() - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503


ModelResponse(id='chatcmpl-1va7n9d9q4jhw0pb9gxz2y', created=1743230883, model='mistral-small-3.1-24b-instruct-2503', object='chat.completion', system_fingerprint='mistral-small-3.1-24b-instruct-2503', choices=[Choices(finish_reason='stop', index=0, message=Message(content='Here are some sources where you can find the latest world news:\n\n- [Yahoo News](https://news.yahoo.com/world/)\n- [AP News](https://apnews.com/world-news)\n- [Mint](https://www.livemint.com/news/world)\n\nWould you like me to provide summaries of specific articles or headlines from these sources?', role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None, 'annotations': None}))], usage=Usage(completion_tokens=76, prompt_tokens=1184, total_tokens=1260, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})

That looks good! We can wrap all of this up into some easier to use agentic logic to keep track of the conversation, execute tools when needed, etc, like so:

In [55]:
import json
from typing import Callable


class Agent:
    def __init__(self, tools: list[Callable]):
        self.tools = tools
        self.function_schemas = get_schemas_openai(tools)
        self.mapping = {tool.__name__: tool for tool in tools}
        self.messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that can use tools to help answer questions "
                    "for the user."
                )
            },
        ]
        
    async def __call__(self, query: str, max_iterations: int = 3) -> str:
        self.messages.append({"role": "user", "content": query})
        i = 0
        while i < max_iterations:
            response = await acompletion(
                model="openai/mistral-small-3.1-24b-instruct-2503",
                messages=self.messages,
                tools=self.function_schemas,
                tool_choice="auto",
                api_key="sk-some-api-key",
                base_url="http://localhost:1234/v1",
            )
            # check if we got a tool call
            if (tool_calls := response.choices[0].message.tool_calls):
                tool_name = tool_calls[0].function.name
                tool_args = json.loads(tool_calls[0].function.arguments)
                tool_call_id = tool_calls[0].id
            else:
                tool_calls = None
                tool_name = None
                tool_args = None
                tool_call_id = None
            # append assistant message
            self.messages.append({
                "role": "assistant",
                "content": response.choices[0].message.content,
                "tool_calls": tool_calls,
                "tool_call_id": tool_call_id
            })
            if tool_calls:
                # if we got a tool call, we execute it and add the message to the conversation
                tool_out = await self.mapping[tool_name](**tool_args)
                self.messages.append({
                    "role": "tool", "content": tool_out, "tool_call_id": tool_call_id
                })
                i += 1
            else:
                # if we didn't get a tool call we assume the iteration is complete
                break
        return self.messages[-1]

In [56]:
agent = Agent([web_search])
out = await agent("tell me about the latest world news")
out

11:26:42 - LiteLLM:INFO: utils.py:3035 - 
LiteLLM completion() model= mistral-small-3.1-24b-instruct-2503; provider = openai
2025-03-29 11:26:42 - LiteLLM - INFO - utils.py:3035 - _check_valid_arg() - 
LiteLLM completion() model= mistral-small-3.1-24b-instruct-2503; provider = openai
2025-03-29 11:26:44 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
11:26:44 - LiteLLM:INFO: cost_calculator.py:588 - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503
2025-03-29 11:26:44 - LiteLLM - INFO - cost_calculator.py:588 - completion_cost() - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503
11:26:44 - LiteLLM:INFO: cost_calculator.py:588 - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503
2025-03-29 11:26:44 - LiteLLM - INFO - cost_calculator.py:588 - completion_cost() - selected model name for cost calc

TypeError: 'NoneType' object is not subscriptable

11:26:56 - LiteLLM:INFO: cost_calculator.py:588 - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503
2025-03-29 11:26:56 - LiteLLM - INFO - cost_calculator.py:588 - completion_cost() - selected model name for cost calculation: openai/mistral-small-3.1-24b-instruct-2503


In [ ]:
out["content"]

---